In [ ]:
import os
import numpy as np
import tensorflow as tf
import pandas as pd
from tqdm import tqdm
import skimage
from PIL import Image
tf.__version__

In [ ]:
# Select the folder with the files
IMAGE_DIR = "/home/cyril/development/data/NoduLoCC2026"
IMG_SIZE = 512

In [ ]:
models = []
models.append('EfficientNetB5_50e.keras')
models.append('EfficientNetB5_best100e_BCE.keras')
models.append('EfficientNetB6_best100_BCE.keras')
models.append('EfficientNetB7_100e_BCE.keras')
MODELS = models

In [ ]:
# Load and normalize an image.
# Input = filename
# Output = numpy array of the image
def load_image(path):
    image = Image.open(path)
    image = np.array(image)
    # Check non "classical" 1024x1024
    if image.shape != (1024, 1024):
        # RGBA case
        if image.shape == (1024, 1024, 4):
            if (image[:, :, 0] == image[:, :, 1]).all() and (image[:, :, 0] == image[:, :, 2]).all() and np.all(image[:, :, 3] == 255):
                image = image[:, :, 0]
        # Non square case
        elif len(image.shape) == 2:
            # Pad
            if image.shape[0] > image.shape[1]:
                image = np.pad(image, ((0, 0), (0, image.shape[0] - image.shape[1])), mode='symmetric')
            elif image.shape[0] < image.shape[1]:
                image = np.pad(image, ((0, image.shape[1] - image.shape[0]), (0, 0)), mode='symmetric')
            # Check padding
            if not (image.shape[0] == image.shape[1]):
                raise ValueError(image.shape)
            # Resize
            image = skimage.transform.resize(image, (1024, 1024), anti_aliasing=True)
        else:
            raise ValueError("ERROR", image.shape, path)
    # Downscale
    image = image.reshape(512, 2, 512, 2).mean(axis=(1, 3)) # downscale // 2
    
    image = (image / 255.0).astype(np.float16)
    return image

In [ ]:
# Load CSV
df = pd.read_csv(f"{IMAGE_DIR}/classification_labels.csv")
# Keep 5636 last example (50/50)
df = df[-2818*2:]
# Keep TEST_SIZE examples
# TEST_SIZE = 512
# df = df.sample(TEST_SIZE)
filenames = df["file_name"].values
labels = df["label"].values
labels = (labels == 'Nodule').astype(np.float32)
print(np.sum(labels)/len(labels))

In [ ]:
X = np.zeros((len(labels), IMG_SIZE, IMG_SIZE, 1), dtype=np.float16)
Y = np.zeros((len(labels), 1), dtype=np.float32)

for i in tqdm(range(len(labels))):
    image = load_image(f'{IMAGE_DIR}/nih_filtered_images/{filenames[i]}')
    label = labels[i]
    
    X[i, :, :, 0] = image
    Y[i, 0] = label

print(X.shape, Y.shape)

In [ ]:
def predict(batch):
    preds = []

    for model_file in MODELS:
        model = tf.keras.models.load_model(model_file)
        pred = model.predict(batch, verbose=0)
        preds.append(pred)

    preds = np.array(preds) # shape: (n_models, batch, n_classes)
    binary_prediction = np.mean(preds, axis=0) > 0.25

    votes = preds > 0.25
    agreement = np.mean(votes == binary_prediction, axis=0)
    variance = np.var(preds, axis=0)
    confidence = agreement * (1 - variance)

    return binary_prediction, confidence, preds

In [ ]:
binary_prediction, confidence_prediction, raw_prediction = predict(X)

In [ ]:
y_pred = binary_prediction[:, 0]
y_true = labels

tp = ((y_pred == 1) & (y_true == 1)).sum()
tn = ((y_pred == 0) & (y_true == 0)).sum()
fp = ((y_pred == 1) & (y_true == 0)).sum()
fn = ((y_pred == 0) & (y_true == 1)).sum()

accuracy = (tp + tn) / len(y_true)
precision = tp / (tp + fp)
recall = tp / (tp + fn)

print(accuracy, precision, recall)
# 0.9874024130589071 0.9824376536705304 0.9925479063165366